In [ ]:
# ================================================================
# AUDITORIA + REPROCESSAMENTO AUTOMÁTICO DOS ANOS FALTANTES
# BASEADO NO SCRIPT: new_rsei_2026_luana_v3.py
#
# O fluxo faz:
# 1) Lê o asset final já existente
# 2) Detecta anos faltantes entre 1985 e 2025 por NM_MUN
# 3) Carrega o script-base sem executar o bloco final de export
# 4) Reutiliza a função build_annual_rsei_image(year, feat)
# 5) Exporta apenas os pares faltantes (NM_MUN x ANO)
# 6) Salva relatórios CSV/XLSX
# Aplicação lê os dois assets (entrada e saída) para listar quais tasks devem rodar novamente
# Mudar no Bloco 8. CARREGAR SCRIPT BASE OUT_ASSET_ROOT = 'projects/rnvuefsppgm/assets/RSEI_Bands_2026_new_new'
# OBS:
# - O arquivo anexado pelo usuário está em:
#   /mnt/data/new_rsei_2026_luana_v3.py
# - O script-base define a lógica de construção do RSEI e o loop
#   de exportação por município/ano. Aqui nós reaproveitamos
#   essa lógica de forma controlada, apenas para os dados faltantes.
# ================================================================

# =========================
# 1. INSTALAÇÃO
# =========================
#!pip -q install earthengine-api geemap pandas openpyxl

# =========================
# 2. IMPORTS
# =========================
import os
import re
import ee
import pandas as pd
from google.colab import drive

# =========================
# 3. AUTENTICAÇÃO EE
# =========================
try:
    #ee.Initialize(project=os.environ.get('EE_PROJECT'))
    ee.Initialize(project=os.environ.get('EE_PROJECT'))
except Exception:
    ee.Authenticate()
    #ee.Initialize(project=os.environ.get('EE_PROJECT'))
    ee.Initialize(project=os.environ.get('EE_PROJECT'))

# =========================
# 4. MONTAR DRIVE (opcional)
# =========================
MONTAR_DRIVE = True
if MONTAR_DRIVE:
    drive.mount('/content/drive')

# =========================
# 5. CONFIGURAÇÕES GERAIS
# =========================

# Assets que serão usados para verificar o que já existe
ASSET_IDS_LEITURA = [
    'projects/ee-rnvuefs/assets/RSEI_Bands_2026_new_new',
    'projects/rnvuefsppgm/assets/RSEI_Bands_2026_new_new'
]

# Asset de destino das novas exportações
ASSET_SALVAR = 'projects/rnvuefsppgm/assets/RSEI_Bands_2026_new_new'

SCRIPT_PATH = '<YOUR_DRIVE_PATH> Notebooks/NEW_RSEI_2026_Luana_v3.py'

ANO_INICIAL = 1985
ANO_FINAL = 2025
ANOS_ESPERADOS = list(range(ANO_INICIAL, ANO_FINAL + 1))

OUT_DIR = '<YOUR_DRIVE_PATH>' if MONTAR_DRIVE else '/content'
os.makedirs(OUT_DIR, exist_ok=True)

RELATORIO_IMAGENS_CSV = os.path.join(OUT_DIR, 'rsei_imagens_existentes.csv')
RELATORIO_RESUMO_CSV = os.path.join(OUT_DIR, 'rsei_resumo_municipios.csv')
RELATORIO_FALTANTES_CSV = os.path.join(OUT_DIR, 'rsei_faltantes_expandido.csv')
RELATORIO_XLSX = os.path.join(OUT_DIR, 'rsei_auditoria_e_faltantes.xlsx')
RELATORIO_TASKS_CSV = os.path.join(OUT_DIR, 'rsei_tasks_reprocessamento.csv')

DRY_RUN = False
TEST_LIMIT = None

In [ ]:
# =========================
# 6. FUNÇÕES AUXILIARES
# =========================
def normalizar_ano(valor):
    if valor is None:
        return None
    try:
        return int(valor)
    except Exception:
        try:
            return int(float(valor))
        except Exception:
            return None

def normalizar_nm_mun(valor):
    if valor is None:
        return None
    return str(valor).strip()

def tentar_ler_como_imagecollection(asset_id):
    ic = ee.ImageCollection(asset_id)
    n = ic.size().getInfo()
    return ic, n

def listar_assets_em_pasta(asset_id):
    resultado = ee.data.listAssets({'parent': asset_id})
    return resultado.get('assets', [])

def obter_registros_via_imagecollection(asset_id):
    ic = ee.ImageCollection(asset_id)

    fc = ic.map(
        lambda img: ee.Feature(
            None,
            {
                'asset_id': img.get('system:id'),
                'system_index': img.get('system:index'),
                'NM_MUN': img.get('NM_MUN'),
                'ANO': img.get('ANO')
            }
        )
    )

    registros = fc.getInfo()['features']
    saida = []
    for f in registros:
        props = f['properties']
        saida.append({
            'asset_id': props.get('asset_id'),
            'system_index': props.get('system_index'),
            'NM_MUN': normalizar_nm_mun(props.get('NM_MUN')),
            'ANO': normalizar_ano(props.get('ANO'))
        })
    return saida

def ler_registros_de_multiplos_assets(asset_ids):
    """
    Lê múltiplos assets e retorna um DataFrame único com os registros existentes.
    Remove duplicidades com base em NM_MUN + ANO.
    """
    todos_registros = []

    for asset_id in asset_ids:
        print('\n' + '-' * 80)
        print(f'Lendo asset: {asset_id}')

        try:
            ic, n = tentar_ler_como_imagecollection(asset_id)
            print(f'Asset lido como ImageCollection com {n} imagens.')
            registros = obter_registros_via_imagecollection(asset_id)
        except Exception as e:
            print('Falha ao ler como ImageCollection. Tentando como pasta...')
            print(f'Motivo: {e}')
            registros = obter_registros_via_listassets(asset_id)
            print(f'Imagens encontradas via listAssets: {len(registros)}')

        # acrescenta coluna com o asset de origem do registro
        for r in registros:
            r['asset_origem'] = asset_id

        todos_registros.extend(registros)

    if len(todos_registros) == 0:
        raise ValueError('Nenhum registro foi encontrado nos assets informados.')

    df = pd.DataFrame(todos_registros)

    # remove duplicidade por município + ano
    # se existir nos dois assets, conta apenas uma vez para auditoria de faltantes
    df = df.drop_duplicates(subset=['NM_MUN', 'ANO']).reset_index(drop=True)

    return df

def gerar_tabelas_de_saida(registros, anos_esperados):
    df = pd.DataFrame(registros)

    if df.empty:
        raise ValueError('Nenhuma imagem foi encontrada no asset informado.')

    df = df.sort_values(by=['NM_MUN', 'ANO', 'system_index'], na_position='last').reset_index(drop=True)

    resumo = []
    faltantes_expandido = []

    for nm_mun, grupo in df.groupby('NM_MUN', dropna=False):
        anos_existentes = sorted({a for a in grupo['ANO'].tolist() if a is not None})
        anos_faltantes = [a for a in anos_esperados if a not in anos_existentes]

        resumo.append({
            'NM_MUN': nm_mun,
            'n_imagens': len(grupo),
            'anos_existentes': ', '.join(map(str, anos_existentes)) if anos_existentes else '',
            'qtd_anos_existentes': len(anos_existentes),
            'anos_faltantes': ', '.join(map(str, anos_faltantes)) if anos_faltantes else '',
            'qtd_anos_faltantes': len(anos_faltantes),
            'completo_1985_2025': 'SIM' if len(anos_faltantes) == 0 else 'NAO'
        })

        for ano in anos_faltantes:
            faltantes_expandido.append({
                'NM_MUN': nm_mun,
                'ANO_FALTANTE': ano
            })

    df_resumo = pd.DataFrame(resumo).sort_values(by=['NM_MUN']).reset_index(drop=True)
    df_faltantes = pd.DataFrame(faltantes_expandido).sort_values(by=['NM_MUN', 'ANO_FALTANTE']).reset_index(drop=True)

    return df, df_resumo, df_faltantes

def sanitize_name(txt):
    txt = '' if txt is None else str(txt)
    return re.sub(r'[^A-Za-z0-9_]', '_', txt)

def load_template_namespace(script_path):
    """
    Carrega o script base sem executar o bloco final de exportação.
    O corte é feito antes do cabeçalho '# EXPORTS'.
    """
    if not os.path.exists(script_path):
        raise FileNotFoundError(f"Script não encontrado: {script_path}")

    with open(script_path, 'r', encoding='utf-8') as f:
        code = f.read()

    marker = '# EXPORTS'
    pos = code.find(marker)
    if pos == -1:
        raise RuntimeError("Não foi encontrado o marcador '# EXPORTS' no script base.")

    code_prefix = code[:pos]

    ns = {}
    exec(code_prefix, ns, ns)
    return ns

def montar_lookup_features(fc_info):
    """
    Cria um dicionário {NM_MUN: feature_json} a partir do getInfo do FC.
    """
    lookup = {}
    for f in fc_info['features']:
        props = f.get('properties', {})
        nm = props.get('NM_MUN')
        if nm is not None:
            lookup[str(nm).strip()] = f
    return lookup



# =========================
# 7. AUDITORIA DOS ASSETS DE LEITURA
# =========================
print('=' * 80)
print('1) AUDITANDO OS ASSETS DE LEITURA')
print('=' * 80)

print('\nAssets usados para leitura dos existentes:')
for a in ASSET_IDS_LEITURA:
    print(f' - {a}')

print(f'\nAsset de destino das novas tasks: {ASSET_SALVAR}')

# Lê e unifica registros dos dois assets
df_imagens = ler_registros_de_multiplos_assets(ASSET_IDS_LEITURA)

# Gera tabelas de resumo/faltantes com base no conjunto unificado
df_imagens, df_resumo, df_faltantes = gerar_tabelas_de_saida(
    df_imagens.to_dict(orient='records'),
    ANOS_ESPERADOS
)

# Salvar relatórios
df_imagens.to_csv(RELATORIO_IMAGENS_CSV, index=False, encoding='utf-8-sig')
df_resumo.to_csv(RELATORIO_RESUMO_CSV, index=False, encoding='utf-8-sig')
df_faltantes.to_csv(RELATORIO_FALTANTES_CSV, index=False, encoding='utf-8-sig')

with pd.ExcelWriter(RELATORIO_XLSX, engine='openpyxl') as writer:
    df_imagens.to_excel(writer, sheet_name='imagens_existentes', index=False)
    df_resumo.to_excel(writer, sheet_name='resumo_municipios', index=False)
    df_faltantes.to_excel(writer, sheet_name='faltantes_expandido', index=False)

print('\nResumo geral:')
print(f'- Total de registros únicos existentes: {len(df_imagens)}')
print(f'- Total de municípios: {len(df_resumo)}')
print(f'- Total de pares faltantes NM_MUN x ANO: {len(df_faltantes)}')

if len(df_faltantes) == 0:
    print('\nNenhum ano faltante encontrado. Nada a reprocessar.')
else:
    print('\nPrimeiros faltantes identificados:')
    print(df_faltantes.head(100))

In [ ]:
# =========================
# 8. CARREGAR SCRIPT BASE
# =========================
print('\n' + '=' * 80)
print('2) CARREGANDO SCRIPT BASE SEM EXECUTAR O BLOCO FINAL DE EXPORT')
print('=' * 80)

ns = load_template_namespace(SCRIPT_PATH)

build_annual_rsei_image = ns['build_annual_rsei_image']
allGeometries = ns['allGeometries']
SCALE = ns['SCALE']

# Guarda só para referência
OUT_ASSET_ROOT_SCRIPT_BASE = ns['OUT_ASSET_ROOT']

# Usa o asset de destino definido no script atual
OUT_ASSET_ROOT = ASSET_SALVAR

print('Script base carregado com sucesso.')
print(f'OUT_ASSET_ROOT do script base (referência): {OUT_ASSET_ROOT_SCRIPT_BASE}')
print(f'OUT_ASSET_ROOT usado nas tasks: {OUT_ASSET_ROOT}')
print(f'SCALE: {SCALE}')


In [ ]:
# =========================
# 9. PREPARAR FEATURES DOS MUNICÍPIOS FALTANTES
# =========================
print('\n' + '=' * 80)
print('3) PREPARANDO FEATURES DOS MUNICÍPIOS FALTANTES')
print('=' * 80)

if len(df_faltantes) == 0:
    print('Sem faltantes. Nada a exportar.')
    tasks = []

else:
    municipios_faltantes = sorted(df_faltantes['NM_MUN'].dropna().unique().tolist())
    print(f'Total de municípios com faltantes: {len(municipios_faltantes)}')

    fc_missing = allGeometries.filter(ee.Filter.inList('NM_MUN', municipios_faltantes))
    fc_missing_info = fc_missing.getInfo()

    feature_lookup = montar_lookup_features(fc_missing_info)

    municipios_sem_feature = sorted(set(municipios_faltantes) - set(feature_lookup.keys()))
    if municipios_sem_feature:
        print('\nATENÇÃO: municípios sem feature correspondente:')
        for m in municipios_sem_feature:
            print(' -', m)

    faltantes_validos = df_faltantes[df_faltantes['NM_MUN'].isin(feature_lookup.keys())].copy()

    if TEST_LIMIT is not None:
        faltantes_validos = faltantes_validos.head(TEST_LIMIT).copy()

    print(f'\nTotal de pares válidos para reprocessamento: {len(faltantes_validos)}')
    # =========================
    # 10. CRIAR E INICIAR EXPORTAÇÕES
    # =========================
    print('\n' + '=' * 80)
    print('4) INICIANDO EXPORTAÇÕES')
    print('=' * 80)

    tasks = []
    relatorio_tasks = []

    for _, row in faltantes_validos.iterrows():
        nm_raw = str(row['NM_MUN']).strip()
        ano = int(row['ANO_FALTANTE'])

        feat_json = feature_lookup[nm_raw]
        feat = ee.Feature(feat_json)

        props = feat_json.get('properties', {})
        nm = sanitize_name(nm_raw)
        cod = str(props.get('COD') or props.get('CD_MUN') or feat_json.get('id') or 'ID')

        try:
            img = build_annual_rsei_image(ano, feat)

            asset_id = f"{OUT_ASSET_ROOT}/{nm}_{cod}_RSEI_{ano}"
            desc = f"EXP_RSEI_{nm}_{ano}"

            task = ee.batch.Export.image.toAsset(
                image=img.toFloat(),
                description=desc,
                assetId=asset_id,
                region=feat.geometry(),
                scale=SCALE,
                maxPixels=1e13
            )

            if not DRY_RUN:
                task.start()

            tasks.append(task)

            relatorio_tasks.append({
                'NM_MUN': nm_raw,
                'ANO': ano,
                'descricao': desc,
                'asset_id': asset_id,
                'assets_leitura': ' | '.join(ASSET_IDS_LEITURA),
                'asset_destino': ASSET_SALVAR,
                'status_inicial': 'READY' if DRY_RUN else 'STARTED'
            })

            print(f'[OK] {nm_raw} | {ano} -> {asset_id}')

        except Exception as e:
            relatorio_tasks.append({
                'NM_MUN': nm_raw,
                'ANO': ano,
                'descricao': None,
                'asset_id': None,
                'status_inicial': f'ERRO: {str(e)}'
            })
            print(f'[ERRO] {nm_raw} | {ano} -> {e}')

    df_tasks = pd.DataFrame(relatorio_tasks)
    df_tasks.to_csv(RELATORIO_TASKS_CSV, index=False, encoding='utf-8-sig')

    print('\n' + '=' * 80)
    print('FINALIZAÇÃO')
    print('=' * 80)
    print(f'Total de tasks registradas: {len(df_tasks)}')
    print(f'Relatório das tasks: {RELATORIO_TASKS_CSV}')
    print('\nRelatórios:')
    print(RELATORIO_IMAGENS_CSV)
    print(RELATORIO_RESUMO_CSV)
    print(RELATORIO_FALTANTES_CSV)
    print(RELATORIO_XLSX)

    if len(tasks) > 0:
        print('\nExemplo para consultar status:')
        print('tasks[0].status()')
